# Perturbations

`HighFidelityEphemerisModel.jl` augments the central two-body term with optional perturbations:

- third-body gravity from other massive bodies
- spherical harmonics of the central body
- solar radiation pressure (SRP)
- atmospheric drag

Mathematical definitions appear in the [Overview](../docs/src/overview.md). This page shows how to enable each term in `SpiceParameters`, `EphemeridesParameters`, or `InterpParameters` and propagate with `OrdinaryDiffEq.jl`.

> **Tip**
>
> Before running the examples, download SPICE kernels and set `ENV["SPICE"]` to your kernel directory. Generic kernels: [https://naif.jpl.nasa.gov/pub/naif/generic_kernels/](https://naif.jpl.nasa.gov/pub/naif/generic_kernels/).

> **Note**
>
> The **first** entry in `naif_ids` is always the central body. All positions and velocities in the state vector are expressed in canonical units (`DU`, `DU/TU`) relative to that body.

## Running this notebook

The next cell activates the environment shipped with the examples (`examples/Project.toml`), which uses the local copy of `HighFidelityEphemerisModel.jl`. Skip it if you would rather use your own environment with `HighFidelityEphemerisModel`, `OrdinaryDiffEq`, and `SPICE` installed.

SPICE kernels are loaded from the directory given by the `SPICE` environment variable, which should contain the `lsk/`, `spk/`, `pck/`, and `fk/` subfolders of the [NAIF generic kernels](https://naif.jpl.nasa.gov/pub/naif/generic_kernels/). Edit `spice_dir` below if your kernels live elsewhere.

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

## Shared setup

The snippets below assume the following preamble (load once per Julia session):

In [ ]:
using LinearAlgebra
using SPICE
using OrdinaryDiffEq
using HighFidelityEphemerisModel

# load SPICE kernels
spice_dir = ENV["SPICE"]
furnsh(joinpath(spice_dir, "lsk", "naif0012.tls"))
furnsh(joinpath(spice_dir, "spk", "de440.bsp"))
furnsh(joinpath(spice_dir, "pck", "gm_de440.tpc"))
furnsh(joinpath(spice_dir, "pck", "pck00011.tpc"))           # IAU_EARTH frame (used for drag)
furnsh(joinpath(spice_dir, "pck", "moon_pa_de440_200625.bpc"))
furnsh(joinpath(spice_dir, "fk", "moon_de440_250416.tf"))

naif_frame = "J2000"
abcorr = "NONE"
et0 = str2et("2026-01-05T00:00:00")

# typical initial state and short propagation window (canonical units)
x0 = [1.05, 0.0, 0.3, 0.5, 1.0, 0.0]

## Third-body perturbations

Third-body accelerations are included automatically for every body listed in `naif_ids` after the central body. Ephemerides are queried from the backend selected by the parameter type.

In [ ]:
naif_ids = ["301", "399", "10"]   # Moon, Earth, Sun (301 = central body)
GMs = [bodvrd(ID, "GM", 1)[1] for ID in naif_ids]
DU = 1e5

parameters = SpiceParameters(
    et0, DU, GMs, naif_ids, naif_frame, abcorr,
)

tspan = (0.0, 6 * 3600 / parameters.TU)
prob = ODEProblem(eom_Nbody!, x0, tspan, parameters)
sol_nbody = solve(prob, Vern7(), reltol=1e-12, abstol=1e-12)
sol_nbody.u[end]

> **Note**
>
> NAIF body IDs: [https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/C/req/naif_ids.html](https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/C/req/naif_ids.html)

## Spherical harmonics

Use the `NbodySH` equations of motion when the gravity field of the central body is not spherical. Provide a gravity model file, the maximum degree `nmax`, and the planet-centered planet-fixed (PCPF) frame name.

In [ ]:
naif_ids = ["301", "399", "10"]
GMs = [bodvrd(ID, "GM", 1)[1] for ID in naif_ids]
DU = 1e5
nmax = 4
filepath_spherical_harmonics = joinpath(
    pkgdir(HighFidelityEphemerisModel),
    "data", "luna", "gggrx_1200l_sha_20x20.tab",
)

parameters = SpiceParameters(
    et0, DU, GMs, naif_ids, naif_frame, abcorr;
    filepath_spherical_harmonics = filepath_spherical_harmonics,
    nmax = nmax,
    frame_PCPF = "MOON_PA",
)

tspan = (0.0, 6 * 3600 / parameters.TU)
prob = ODEProblem(eom_NbodySH!, x0, tspan, parameters)
sol_SH = solve(prob, Vern7(), reltol=1e-12, abstol=1e-12)

# effect of the spherical harmonics on the final position, in km
norm(sol_SH.u[end][1:3] - sol_nbody.u[end][1:3]) * DU

> **Warning**
>
> Spherical harmonics require `eom_NbodySH!` or a backend-specific `eom_NbodySH_*` function. The `Nbody` variants do not evaluate harmonic terms.

## Ephemerides.jl backend

`EphemeridesParameters` uses Julia-native ephemeris queries instead of direct SPICE ephemeris calls at runtime. Pass either an existing `ephemerides_provider` or a list of ephemeris files through `ephemerides_files`. For spherical harmonics, also provide the binary PCK needed by the body-fixed frame transformation.

In [ ]:
paths = [
    joinpath(spice_dir, "spk", "de440.bsp"),
    joinpath(spice_dir, "pck", "moon_pa_de440_200625.bpc"),
]

parameters = EphemeridesParameters(
    et0, DU, GMs, naif_ids, naif_frame, abcorr;
    filepath_spherical_harmonics = filepath_spherical_harmonics,
    nmax = nmax,
    frame_PCPF = "MOON_PA",
    include_srp = true,
    ephemerides_files = paths,
)

tspan = (0.0, 6 * 3600 / parameters.TU)
prob = ODEProblem(eom_NbodySH!, x0, tspan, parameters)
sol_ephemerides = solve(prob, Vern7(), reltol=1e-12, abstol=1e-12)
sol_ephemerides.u[end]

> **Note**
>
> The Ephemerides.jl backend supports third-body gravity, spherical harmonics, SRP, and drag. Frame transforms such as `"J2000"` to `"MOON_PA"` are handled through `FrameTransformations.jl`. For common DE440 body pairs, the helper routines also try segment-chain fallbacks through the solar-system barycenter and Earth-Moon barycenter when a direct pair is unavailable.

## Solar radiation pressure

Enable the cannonball SRP model with `include_srp = true` and spacecraft properties `srp_Cr`, `srp_Am`. The Sun (NAIF ID `"10"`) **must** appear in `naif_ids` so its ephemeris is available.

In [ ]:
naif_ids = ["301", "399", "10"]
GMs = [bodvrd(ID, "GM", 1)[1] for ID in naif_ids]
DU = 1e5

parameters = SpiceParameters(
    et0, DU, GMs, naif_ids, naif_frame, abcorr;
    include_srp = true,
    srp_Cr = 1.15,
    srp_Am = 0.002,
    srp_P0 = 4.56e-6,
)

tspan = (0.0, 6 * 3600 / parameters.TU)
prob = ODEProblem(eom_Nbody!, x0, tspan, parameters)
sol_SRP = solve(prob, Vern7(), reltol=1e-12, abstol=1e-12)

# effect of SRP on the final position, in km
norm(sol_SRP.u[end][1:3] - sol_nbody.u[end][1:3]) * DU

> **Warning**
>
> If `include_srp = true` but `"10"` is missing from `naif_ids`, parameter construction throws an error.

## Atmospheric drag

Drag uses a quadratic law with density supplied by a callback `f_density(et, r_km)` returning $\rho$ in kg/m³. The atmosphere is modeled as co-rotating with angular rate `omega_atm` (default: Earth sidereal rate about the $z$-axis).

When drag is enabled, the EOM transform the spacecraft position from `naif_frame` to `frame_PCPF` before calling `f_density`. Custom callbacks must therefore expect **`r_km` in `frame_PCPF` (km)**, not in the inertial frame. Set `frame_PCPF` (e.g. `"IAU_EARTH"` for Earth drag; this frame requires a text PCK such as `pck00011.tpc` to be loaded).

A built-in Harris–Priester table is available via `harris_priester_f_density(R_earth_km)`. The argument `R_earth_km` is the **Earth reference radius in km** used to form altitude as `norm(r_km) - R_earth_km`; it is not the canonical distance unit `DU` (those may differ if you choose a different `DU` for scaling). The Jacchia–Roberts model is available via `jacchia_roberts_f_density` (constant F10.7 / F10.7a / Kp; SPICE geodetics). Drag is supported by `eom_Nbody!` and `eom_NbodySH!` for all parameter backends, including Ephemerides.jl.

In [ ]:
naif_ids = ["399", "10"]   # Earth-centered; Sun for optional third-body / SRP
GMs = [bodvrd(ID, "GM", 1)[1] for ID in naif_ids]
DU = 6378.0
R_earth_km = 6378.0        # Earth radius in km (for Harris–Priester altitude; not DU)

f_density = harris_priester_f_density(R_earth_km; use_min=true)

parameters = SpiceParameters(
    et0, DU, GMs, naif_ids, naif_frame, abcorr;
    frame_PCPF = "IAU_EARTH",
    include_drag = true,
    drag_Cd = 2.2,
    drag_Am = 0.01,
    f_density = f_density,
)

x0_earth = [1.05, 0.0, 0.01, 0.0, 1.0, 0.0]
tspan = (0.0, 2 * 86400 / parameters.TU)
prob = ODEProblem(eom_Nbody!, x0_earth, tspan, parameters)
sol_drag = solve(prob, Vern7(), reltol=1e-12, abstol=1e-12)
sol_drag.u[end]

`f_density` is expected to be a function with signature `f_density(et, r_km)`, where `et` is the epoch in ephemeris seconds and `r_km` is the position vector in **`frame_PCPF`**, km.

> **Note**
>
> `include_drag = true` requires both `f_density` and `frame_PCPF` to be provided. Coefficient `k_drag` is computed internally from `drag_Cd` and `drag_Am` in SI units.

## Combining perturbations

Flags compose in a single parameter object. The example below propagates a low Earth orbit with the legacy interpolated backend (compatible with `EnsembleThreads` and `ForwardDiff`), and enables lunar and solar third-body gravity, Earth spherical harmonics, SRP, and drag together. For new non-SPICE workflows, prefer `EphemeridesParameters` when the needed kernels are available through Ephemerides.jl.

In [ ]:
naif_ids = ["399", "301", "10"]     # Earth-centered, with Moon & Sun as third bodies
GMs = [bodvrd(ID, "GM", 1)[1] for ID in naif_ids]
R_earth_km = 6378.0
DU = R_earth_km
nmax = 4
filepath_spherical_harmonics = joinpath(
    pkgdir(HighFidelityEphemerisModel),
    "data", "earth", "GGM03S.tab",
)

etf = et0 + 1 * 86400.0
interpolate_ephem_span = [et0, etf]
interpolation_time_step = 60.0

parameters = InterpParameters(
    et0, DU, GMs, naif_ids, naif_frame, abcorr;
    interpolate_ephem_span = interpolate_ephem_span,
    interpolation_time_step = interpolation_time_step,
    filepath_spherical_harmonics = filepath_spherical_harmonics,
    nmax = nmax,
    frame_PCPF = "IAU_EARTH",
    include_srp = true,
    srp_Cr = 1.15,
    srp_Am = 0.002,
    include_drag = true,
    drag_Cd = 2.2,
    drag_Am = 0.01,
    f_density = harris_priester_f_density(R_earth_km; use_min=true),
)

# circular orbit at 400 km altitude, 51.6 deg inclination
r0 = (R_earth_km + 400.0) / DU
v0 = sqrt(1 / r0)
inc = deg2rad(51.6)
x0_leo = [r0, 0.0, 0.0, 0.0, v0 * cos(inc), v0 * sin(inc)]

tspan = (0.0, 6 * 3600 / parameters.TU)
prob = ODEProblem(eom_NbodySH!, x0_leo, tspan, parameters)
sol_combined = solve(prob, Vern8(), reltol=1e-12, abstol=1e-12)

# final altitude, in km
norm(sol_combined.u[end][1:3]) * DU - R_earth_km

> **Warning**
>
> `InterpParameters` only provides ephemerides and frame transformations within `interpolate_ephem_span`, so make sure this window covers the entire propagation. When the PCPF frame rotates quickly (e.g. `"IAU_EARTH"`), use a small enough `interpolation_time_step` to resolve the rotation.

## Choosing an equation of motion

| Model | Perturbations included | SPICE | Ephemerides.jl | Legacy interpolation |
|-------|------------------------|-------|----------------|----------------------|
| `Nbody` | third-body, optional SRP & drag | `SpiceParameters` | `EphemeridesParameters` | `InterpParameters` |
| `NbodySH` | above + spherical harmonics | `SpiceParameters` | `EphemeridesParameters` | `InterpParameters` |

See the full function list and STM options in the [Overview](../docs/src/overview.md).

| Use case | Recommended EOM |
|----------|-----------------|
| High accuracy, few propagations | `SpiceParameters` + `eom_NbodySH!` |
| Julia-native ephemerides | `EphemeridesParameters` + `eom_NbodySH!` |
| Legacy interpolation / sensitivities | `InterpParameters` + `eom_NbodySH!` |
| No harmonics needed | `eom_Nbody!` |

For IVP setup and STM propagation, see [Basics](basics.ipynb). For Jacobians and Hessians, see [Jacobians & Hessians](jacobians_hessians.ipynb).